# Predicting Stellar Class from Data

This notebook trains models to predict stellar class for the Predicting Stellar Class challenge, a kaggle competition using synthetic data based on observational data. The synthetic data are published on kaggle:

Yao Yan, Walter Reade, Elizabeth Park. Predicting Stellar Class. https://kaggle.com/competitions/playground-series-s6e6, 2026. Kaggle.

In [1]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('playground-series-s6e6')

print("Path to competition files:", path)

Path to competition files: /Users/jaquesgillis/.cache/kagglehub/competitions/playground-series-s6e6


In [2]:
from setup import * 
import os

os.listdir(path)

['test.csv', 'train.csv', 'sample_submission.csv']

In [3]:
df = pd.read_csv(path + '/train.csv')

In [4]:
df.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [5]:
# Preprocess the data

X = df.copy()
X = X.drop(columns=['id'])

# class lists are the columns we will transform
# value lists will give us the names for the transformed columns
ordinal_class = ['class']
ordinal_values = X['class'].unique()
spectral_class = ['spectral_type']
galaxy_class = ['galaxy_population']
hot_spectral_values = X['spectral_type'].unique()
hot_galaxy_values = X['galaxy_population'].unique()

# ordinal encode the target, one hot encode the other two
one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ordinal = OrdinalEncoder(categories=[ordinal_values])

# one-hot encode
hot_spectral_columns = one_hot.fit_transform(X[spectral_class])
hot_galaxy_columns = one_hot.fit_transform(X[galaxy_class])
new_spectral_columns = pd.DataFrame(hot_spectral_columns)
new_spectral_columns.columns = hot_spectral_values
new_galaxy_columns = pd.DataFrame(hot_galaxy_columns)
new_galaxy_columns.columns = hot_galaxy_values

ordinal_column = ordinal.fit_transform(X[ordinal_class])
new_ordinal_column = pd.DataFrame(ordinal_column)
new_ordinal_column.columns = ['class']

X = pd.concat([X.drop(columns=ordinal_class+spectral_class+galaxy_class, axis=1), new_spectral_columns, new_galaxy_columns], axis=1)
y = ordinal_column

# TODO: 
Let's add columns for O, B, A, etc., since these might be in the test data.
Also, consider dropping 'galaxy_population' or somehow protecting your model from unseen galaxy populations.

### Baseline model

Let's just train a random forest model on the features we have here as a baseline.

In [7]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (577347, 14)
y shape: (577347, 1)


In [8]:
# Using random forest with n=100, max depth = 6

rf_model = RandomForestClassifier(n_estimators=100, max_depth=6)
train_X, val_X, train_y, val_y = train_test_split(X, y)
rf_model.fit(train_X, train_y)
val_predictions = rf_model.predict(val_X)
print(f"Baseline balanced accuracy score: {balanced_accuracy_score(val_y, val_predictions)}")

/opt/miniconda3/envs/kaggle-astro/lib/python3.10/site-packages/sklearn/base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Baseline balanced accuracy score: 0.8988697327186556


### Comment

Our baseline is high, which is good and bad news. It's good news, because without doing any work, we can already predict stellar class with good accuracy. It's bad news because we have a small window in which to make improvements, and even making small improvements is likely to be hard.